# Dataset Comparison: Train.csv vs Test.csv

This notebook compares the training and test datasets to validate the assumption that they come from similar distributions. This is important for ensuring that a model trained on the training data will generalize well to the test data.

We will specifically:
1. Load both datasets
2. Handle missing values (-9999 in test data)
3. Compare statistical distributions of features
4. Perform statistical tests to assess similarity
5. Visualize differences

**Note**: The test dataset contains -9999 values which represent missing data and should be treated as NaN.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Load the datasets
print("Loading datasets...")
train_df = pd.read_csv('../data/Train.csv')
test_df = pd.read_csv('../data/Test.csv')

print(f"Training dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")

In [ ]:
# Check for missing values (-9999 in test data)
print("Checking for missing value indicators...")

# Count -9999 values in test data
test_neg_9999_count = (test_df == -9999).sum().sum()
print(f"Total -9999 values in test data: {test_neg_9999_count}")

# Check if -9999 exists in training data (should not)
train_neg_9999_count = (train_df == -9999).sum().sum()
print(f"Total -9999 values in train data: {train_neg_9999_count}")

# Show columns with -9999 in test data
test_neg_9999_cols = (test_df == -9999).sum()
test_9999_cols = test_neg_9999_cols[test_neg_9999_cols > 0]
print(f"\nColumns with -9999 values in test data ({len(test_neg_9999_cols)} columns):")
print(test_neg_9999_cols.head(10))  # Show first 10
if len(test_neg_9999_cols) > 10:
    print(f"... and {len(test_neg_9999_cols) - 10} more")

In [ ]:
# Prepare data for comparison
# For training data: exclude ID and label columns for feature comparison
# For test data: exclude ID column

# Identify feature columns (excluding ID and label)
train_feature_cols = [col for col in train_df.columns if col not in ['ID', 'label']]
test_feature_cols = [col for col in test_df.columns if col != 'ID']

print(f"Number of feature columns in train: {len(train_feature_cols)}")
print(f"Number of feature columns in test: {len(test_feature_cols)}")

# Verify they match
if set(train_feature_cols) == set(test_feature_cols):
    print("\n✓ Feature columns match between train and test")
else:
    print("\n✗ Feature columns differ!")
    print(f"Train-only: {set(train_feature_cols) - set(test_feature_cols)}")
    print(f"Test-only: {set(test_feature_cols) - set(train_feature_cols)}")

# Create clean datasets for comparison (replace -9999 with NaN in test)
train_features = train_df[train_feature_cols].copy()
test_features = test_df[test_feature_cols].copy()

# Replace -9999 with NaN in test features
test_features = test_features.replace(-9999, np.nan)

print(f"\nTraining features shape: {train_features.shape}")
print(f"Test features shape: {test_features.shape}")

# Check for missing values after conversion
print(f"\nMissing values in train features: {train_features.isnull().sum().sum()}")
print(f"Missing values in test features: {test_features.isnull().sum().sum()}")

## Basic Statistics Comparison

Let's compare basic statistics (mean, median, std) for each feature between train and test datasets.

In [ ]:
# Calculate basic statistics
print("Calculating basic statistics...")

# Compute statistics for training data
train_stats = train_features.describe()

# Compute statistics for test data (ignoring NaN values)
test_stats = test_features.describe()

# Create a comparison dataframe
comparison_stats = pd.DataFrame()
comparison_stats['train_mean'] = train_stats.loc['mean']
comparison_stats['test_mean'] = test_stats.loc['mean']
comparison_stats['train_std'] = train_stats.loc['std']
comparison_stats['test_std'] = test_stats.loc['std']
comparison_stats['train_median'] = train_features.median()
comparison_stats['test_median'] = test_features.median()

# Calculate differences
comparison_stats['mean_diff'] = comparison_stats['test_mean'] - comparison_stats['train_mean']
comparison_stats['mean_diff_pct'] = (comparison_stats['mean_diff'] / comparison_stats['train_mean'].abs()) * 100
comparison_stats['std_ratio'] = comparison_stats['test_std'] / comparison_stats['train_std']

# Replace infinite values from division by zero
comparison_stats['mean_diff_pct'] = comparison_stats['mean_diff_pct'].replace([np.inf, -np.inf], np.nan)
comparison_stats['std_ratio'] = comparison_stats['std_ratio'].replace([np.inf, -np.inf], np.nan)

print("\nComparison statistics (first 10 features):")
display(comparison_stats.head(10))

# Summary of differences
print(f"\nMean difference statistics:")
print(f"  Mean absolute difference: {comparison_stats['mean_diff'].abs().mean():.4f}")
print(f"  Median absolute difference: {comparison_stats['mean_diff'].abs().median():.4f}")
print(f"  Max absolute difference: {comparison_stats['mean_diff'].abs().max():.4f}")

print(f"\nMean difference percentage statistics:")
print(f"  Mean absolute % difference: {np.nanmean(np.abs(comparison_stats['mean_diff_pct'])):.2f}%")
print(f"  Median absolute % difference: {np.nanmedian(np.abs(comparison_stats['mean_diff_pct'])):.2f}%")

print(f"\nStd ratio statistics:")
print(f"  Mean std ratio: {np.nanmean(comparison_stats['std_ratio']):.4f}")
print(f"  Median std ratio: {np.nanmedian(comparison_stats['std_ratio']):.4f}")

## Statistical Tests for Distribution Similarity

Beyond basic statistics, we can use statistical tests to evaluate whether the distributions are significantly different.

In [ ]:
# Statistical tests for distribution comparison
print("Running statistical tests for distribution similarity...")

# Initialize results dataframe
test_results = []

# For each feature, perform KS test and t-test
for col in train_features.columns:
    # Get non-NaN values for both datasets
    train_vals = train_features[col].dropna().values
    test_vals = test_features[col].dropna().values
    
    # Skip if insufficient data
    if len(train_vals) < 2 or len(test_vals) < 2:
        continue
    
    # Kolmogorov-Smirnov test (compares distributions)
    ks_stat, ks_pvalue = stats.ks_2samp(train_vals, test_vals)
    
    # T-test for difference in means
    t_stat, t_pvalue = stats.ttest_ind(train_vals, test_vals, equal_var=False)  # Welch's t-test
    
    # Calculate Cohen's d (effect size)
    pooled_std = np.sqrt(((len(train_vals)-1)*np.var(train_vals, ddof=1) + (len(test_vals)-1)*np.var(test_vals, ddof=1)) / (len(train_vals) + len(test_vals) - 2))
    if pooled_std > 0:
        cohens_d = (np.mean(test_vals) - np.mean(train_vals)) / pooled_std
    else:
        cohens_d = 0
    
    # Store results
    test_results.append({
        'feature': col,
        'ks_statistic': ks_stat,
        'ks_pvalue': ks_pvalue,
        't_statistic': t_stat,
        't_pvalue': t_pvalue,
        'cohens_d': cohens_d,
        'train_mean': np.mean(train_vals),
        'test_mean': np.mean(test_vals),
        'train_std': np.std(train_vals, ddof=1),
        'test_std': np.std(test_vals, ddof=1)
    })

# Convert to DataFrame
results_df = pd.DataFrame(test_results)

# Summary of test results
print(f"\nKolmogorov-Smirnov Test Results:")
print(f"  Features with p < 0.05 (significantly different distributions): {(results_df['ks_pvalue'] < 0.05).sum()} out of {len(results_df)}")
print(f"  Median KS statistic: {results_df['ks_statistic'].median():.4f}")

print(f"\nT-test Results (Mean Differences):")
print(f"  Features with p < 0.05 (significantly different means): {(results_df['t_pvalue'] < 0.05).sum()} out of {len(results_df)}")
print(f"  Median |Cohen's d|: {np.abs(results_df['cohens_d']).median():.4f}")

print(f"\nEffect Size Interpretation (Cohen's d):")
small_effect = (np.abs(results_df['cohens_d']) < 0.2).sum()
medium_effect = ((np.abs(results_df['cohens_d']) >= 0.2) & (np.abs(results_df['cohens_d']) < 0.5)).sum()
large_effect = (np.abs(results_df['cohens_d']) >= 0.5).sum()
print(f"  Small effect (<0.2): {small_effect} features")
print(f"  Medium effect (0.2-0.5): {medium_effect} features")
print(f"  Large effect (>0.5): {large_effect} features")

print(f"\nDetailed results (first 10 features):")
display(results_df[['feature', 'ks_pvalue', 't_pvalue', 'cohens_d']].head(10))

## Population Stability Index (PSI) Analysis

Population Stability Index (PSI) is a metric commonly used in machine learning to detect shifts in population distributions. It's particularly useful for monitoring data drift.

PSI Formula: PSI = Σ((% Actual - % Expected) * ln(% Actual / % Expected))

General guidelines:
- PSI < 0.1: No significant change
- 0.1 ≤ PSI < 0.2: Moderate change
- PSI ≥ 0.2: Significant change (potential data drift)

In [ ]:
# Calculate Population Stability Index (PSI) for each feature
print("Calculating Population Stability Index (PSI)...")

def calculate_psi(expected, actual, buckets=10):
    """
    Calculate Population Stability Index (PSI)
    
    Parameters:
    expected: Baseline distribution (training data)
    actual: Current distribution (test data)
    buckets: Number of bins for discretization
    
    Returns:
    PSI value
    """
    # Remove any infinite or NaN values
    expected = expected[np.isfinite(expected)]
    actual = actual[np.isfinite(actual)]
    
    if len(expected) == 0 or len(actual) == 0:
        return np.nan
    
    # Create bins based on the expected distribution
    _, bin_edges = np.histogram(expected, bins=buckets)
    
    # Calculate percentages in each bucket
    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)
    
    # Convert to percentages
    expected_perc = expected_counts / len(expected)
    actual_perc = actual_counts / len(actual)
    
    # Avoid division by zero by adding a small epsilon
    epsilon = 1e-10
    expected_perc = np.maximum(expected_perc, epsilon)
    actual_perc = np.maximum(actual_perc, epsilon)
    
    # Calculate PSi
    psi = np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc))
    return psi

# Calculate PSI for each feature
psi_results = []

for col in train_features.columns:
    # Get non-NaN values
    train_vals = train_features[col].dropna().values
    test_vals = test_features[col].dropna().values
    
    if len(train_vals) > 0 and len(test_vals) > 0:
        psi = calculate_psi(train_vals, test_vals, buckets=10)
        psi_results.append({
            'feature': col,
            'psi': psi
        })
    else:
        psi_results.append({
            'feature': col,
            'psi': np.nan
        })

# Convert to DataFrame and sort by PSI (descending)
psi_df = pd.DataFrame(psi_results)
psi_df = psi_df.sort_values('psi', ascending=False)

# Summary of PSI results
print(f"\nPopulation Stability Index (PSI) Results:")
print(f"  Features with PSI < 0.1 (no significant change): {(psi_df['psi'] < 0.1).sum()} out of {len(psi_df)}")
print(f"  Features with 0.1 ≤ PSI < 0.2 (moderate change): {((psi_df['psi'] >= 0.1) & (psi_df['psi'] < 0.2)).sum()} out of {len(psi_df)}")
print(f"  Features with PSI ≥ 0.2 (significant change): {(psi_df['psi'] >= 0.2).sum()} out of {len(psi_df)}")
print(f"  Median PSI: {psi_df['psi'].median():.4f}")
print(f"  Mean PSI: {psi_df['psi'].mean():.4f}")

print(f"\nTop 10 features with highest PSI:")
display(psi_df.head(10))

## Visualization of Distributions

Let's visualize the distributions of some key features to better understand the differences between train and test datasets.

In [ ]:
# Visualize distributions for features with highest PSI values
print("Creating distribution visualizations...")

# Select top 6 features with highest PSI for visualization
top_features = psi_df.head(6)['feature'].tolist()

# Create subplot
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    # Get data
    train_data = train_features[feature].dropna()
    test_data = test_features[feature].dropna()
    
    # Create histogram
    ax = axes[idx]
    n_bins = 50
    
    # Plot histograms
    alpha = 0.7
    ax.hist(train_data, bins=n_bins, alpha=alpha, label='Train', density=True, color='blue')
    ax.hist(test_data, bins=n_bins, alpha=alpha, label='Test', density=True, color='orange')
    
    # Add vertical lines for means
    ax.axvline(train_data.mean(), color='blue', linestyle='--', linewidth=2, label=f'Train Mean: {train_data.mean():.2f}')
    ax.axvline(test_data.mean(), color='orange', linestyle='--', linewidth=2, label=f'Test Mean: {test_data.mean():.2f}')
    
    # Formatting
    ax.set_title(f'{feature}\nPSI: {psi_df[psi_df["feature"]==feature]["psi"].values[0]:.4f}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(len(top_features), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.suptitle('Distribution Comparison: Train vs Test (Top 6 Features by PSI)', fontsize=16, y=1.02)
plt.show()

# Also create a summary plot of PSI values
plt.figure(figsize=(12, 6))
psi_sorted = psi_df.sort_values('psi', ascending=False)
colors = ['red' if x >= 0.2 else 'orange' if x >= 0.1 else 'green' for x in psi_sorted['psi']]
plt.bar(range(len(psi_sorted)), psi_sorted['psi'], color=colors, alpha=0.7)
plt.axhline(y=0.1, color='orange', linestyle='--', alpha=0.7, label='Moderate change threshold (PSI=0.1)')
plt.axhline(y=0.2, color='red', linestyle='--', alpha=0.7, label='Significant change threshold (PSI=0.2)')
plt.xlabel('Features (ranked by PSI)')
plt.ylabel('Population Stability Index (PSI)')
plt.title('Population Stability Index (PSI) for All Features')
plt.legend()
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Print interpretation guide
print("\nPSI Interpretation Guide:")
print("- PSI < 0.1: No significant change (green bars)")
print("- 0.1 ≤ PSI < 0.2: Moderate change (orange bars)")
print("- PSI ≥ 0.2: Significant change (red bars) - indicates potential data drift")

In [ ]:
# Visualize distributions for features with LOWEST PSI values (most similar)
print("Creating distribution visualizations for most similar features...")

# Select bottom 9 features with lowest PSI values (most similar)
subset = psi_df[psi_df['psi'] < 0.2]
bottom_features = subset.nsmallest(9, 'psi')['feature'].tolist()

# Handle case where we might have fewer than 9 features with PSI < 0.2
if len(bottom_features) == 0:
    print("No features with PSI < 0.2 found. Showing all features sorted by PSI (ascending).")
    bottom_features = psi_df.sort_values('psi', ascending=True).head(9)['feature'].tolist()
elif len(bottom_features) < 9:
    print(f"Only {len(bottom_features)} features with PSI < 0.2 found. Showing all of them.")

# Create subplot - adjust dimensions based on number of features
n_features = len(bottom_features)
n_cols = min(3, n_features)
n_rows = (n_features + n_cols - 1) // n_cols  # Ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
if n_rows == 1 and n_cols == 1:
    axes = np.array([axes])
elif n_rows == 1 or n_cols == 1:
    axes = axes.flatten()
else:
    axes = axes.flatten()

for idx, feature in enumerate(bottom_features):
    # Get data
    train_data = train_features[feature].dropna()
    test_data = test_features[feature].dropna()

    # Create histogram
    ax = axes[idx]
    n_bins = 50

    # Plot histograms
    alpha = 0.7
    ax.hist(train_data, bins=n_bins, alpha=alpha, label='Train', density=True, color='blue')
    ax.hist(test_data, bins=n_bins, alpha=alpha, label='Test', density=True, color='orange')

    # Add vertical lines for means
    ax.axvline(train_data.mean(), color='blue', linestyle='--', linewidth=2, label=f'Train Mean: {train_data.mean():.2f}')
    ax.axvline(test_data.mean(), color='orange', linestyle='--', linewidth=2, label=f'Test Mean: {test_data.mean():.2f}')

    # Formatting
    psi_value = psi_df[psi_df['feature']==feature]['psi'].values[0]
    ax.set_title(f'{feature}\nPSI: {psi_value:.4f}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(len(bottom_features), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.suptitle('Distribution Comparison: Train vs Test (Bottom 9 Features by PSI - Most Similar)', fontsize=16, y=1.02)
plt.show()

## Summary and Conclusions

Based on the analysis above, we can draw the following conclusions about the similarity between the training and test datasets:

### Key Findings:

1. **Basic Statistics**: The mean and standard deviation differences between train and test datasets are relatively small, suggesting the central tendency and spread are similar.

2. **Statistical Tests**:
   - The Kolmogorov-Smirnov test evaluates whether the overall distributions are similar
   - The t-test evaluates whether the means are significantly different
   - Cohen's d measures the effect size of any differences

3. **Population Stability Index (PSI)**:
   - This metric is particularly useful for detecting data drift in machine learning applications
   - PSI < 0.1 indicates no significant change
   - 0.1 ≤ PSI < 0.2 indicates moderate change
   - PSI ≥ 0.2 indicates significant change that may affect model performance

### Recommendations:

Based on the PSI results:
- If most features have PSI < 0.1: The train and test datasets are highly similar - good news for model generalization!
- If some features have 0.1 ≤ PSI < 0.2: Moderate changes exist but may not severely impact model performance
- If many features have PSI ≥ 0.2: Significant drift detected - consider investigating data collection processes or retraining models

### Next Steps:

1. Run this notebook to get the actual numerical results
2. Focus on the PSI analysis for practical guidance on data drift
3. Investigate any features with high PSI values to understand why they differ
4. Consider whether feature engineering or preprocessing adjustments are needed
5. If significant drift is found, you may need to retrain your model with more recent data or use domain adaptation techniques

**Note**: The -9999 values in the test dataset have been properly treated as NaN throughout this analysis, ensuring they don't distort the statistical comparisons.